# Tool streaming — Claude API (summary)

## Overview
- Tool streaming: streaming mode where Claude emits message content and tool-related events incrementally so clients can display and/or act on partial outputs in real time.
- Use case: observe and/or start processing tool calls as they are being generated (e.g., begin validating or executing inputs before the final assistant message completes).

## Key events to handle
- `ContentBlockDelta`: incremental chunk of a content block (may be text or a tool-related block). Treat as the primary streaming chunk for content.
- `input_json`: the more specific event carrying the planned JSON payload for a tool use. It can arrive as partial JSON fragments or as a final validated object depending on mode.
- Other useful events: `content_block_start` / `content_block_stop` (demarcate blocks), text deltas (plain message text), and `tool_result` blocks returned after tool execution.

## How chunking and validation work (important nuance)
- Message text chunks arrive similarly to normal streaming — small deltas you can render progressively.
- Tool inputs are handled differently in default (non-fine-grained) streaming:
  - Claude performs internal validation per high-level key of the planned `input_json` before emitting it.
  - For each high-level key (e.g., `meta`, `abstract`, `options`), Claude will gather all chunks for that key, validate that key internally, and only then emit that key's value to you.
  - As a result, tool input emission may appear batched or delayed compared to text deltas — you will not necessarily receive each character/byte as it is generated.
  - This design reduces the chance of receiving malformed JSON from Claude in normal streaming, but it introduces latency between the first hint of a tool call and the arrival of its usable inputs.

## Fine-grained tooling (skip validation, faster streaming)
- Fine-grained tooling disables Claude's per-key validation and allows tool inputs to stream as incremental chunks, similar to text.
- How to enable:
  - Use new parameters such as `betas` to opt into beta features (e.g., pass a fine-grained tool-streaming beta name).
  - Use `tool_choice` to request or force specific tool usage (note: forcing a tool affects other Claude behaviors — read docs).
- Effects:
  - Tool inputs now arrive in smaller partial chunks; you can start parsing/processing earlier and display near-real-time progress.
  - No automatic validation is performed by Claude — you receive raw fragments and must validate/assemble them yourself.

## Parameters / flags to know
- `betas`: opt-in array enabling beta features (e.g., fine-grained tool streaming).
- `tool_choice`: can force Claude to use a particular tool or type of tool. Forcing tools can disable or change other assistant behaviors (e.g., extended deliberation or "extending thinking").
- `tools`: tool schema metadata you provide so Claude can emit tool uses in a validated schema form (used in normal mode).
- `tool_choice` + `betas` combination is a powerful lever; consult docs before combining them.

## Practical implications & recommended client behavior
- Default streaming (with validation):
  - Expect delays for tool inputs. Do not assume immediate arrival of `input_json`.
  - Use `content_block_start` / `content_block_stop` to know when a block begins/ends.
  - Validate final JSON (Claude already validated per-key), then run your tool.
- Fine-grained streaming:
  - You will receive partial JSON tokens/chunks. You must:
    - Assemble chunks for a given tool use (track `tool_use_id` or block id).
    - Perform robust incremental JSON parsing or buffering (because fragments may split tokens).
    - Apply your own schema validation before executing the tool.
  - Advantage: lower latency and ability to provide near-real-time UX; disadvantage: increased responsibility for correctness.
- Error handling:
  - Be prepared for `tool_result` blocks that indicate `is_error: True` or stringified error messages.
  - On partial JSON parse failures, buffer until the tool signals block end or a final `input_json` chunk arrives.

## Implementation notes (practical checklist)
- Identify blocks by block id / `tool_use_id` to correlate chunks → assembly → validation → execution.
- Use a streaming JSON parser or incremental buffering strategy:
  - For fine-grained mode: parse partial tokens when possible; if parsing fails, keep buffering until syntactically correct.
  - For default mode: you can often expect whole-object emission for each validated key, but still handle partial cases defensively.
- Distinguish content types: text vs `tool_use` vs `tool_result`; don't treat `tool_use` tokens as plain text.
- If you force a `tool_choice`, test flows where Claude's "thinking" extensions are needed — forcing may bypass those features.
- Log timing and chunk boundaries during development to understand validation-induced delays.

## Best practices & safety
- Prefer default validated streaming in production if you want Claude to pre-validate complex inputs and reduce malformed submissions.
- Use fine-grained tooling only when low latency and real-time feedback are necessary and you can guarantee safe validation on the client side.
- Always validate incoming tool inputs against your schema before invoking side-effecting operations (database writes, network calls).
- Implement idempotency and retries for tool executions where partial or repeated tool inputs might produce duplicate attempts.
- Provide user-facing progress indicators that reflect the difference between "Claude composing" and "tool input available".

## Quick mental model
- Default streaming: Claude = validates keys → emits grouped/validated tool inputs → you execute.
- Fine-grained streaming: Claude = streams raw chunks quickly → you assemble & validate → you execute.

## Summary
- Tool streaming lets you observe tool calls incrementally but the streaming semantics differ for tool inputs vs text.
- Claude's internal per-key validation in default mode causes input emission to be grouped/delayed.
- Fine-grained tooling removes Claude's validation and enables low-latency, chunked tool input streaming — but pushes validation responsibility to you.
- Use the `betas`, `tools`, and `tool_choice` parameters intentionally; test interactions (e.g., forced tools vs extended thinking) before deploying.

In [7]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [8]:
# Helper functions


def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [9]:
# Tool definition
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)
save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs):
    return "Article saved!"


In [10]:
# Tool Running
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [11]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [12]:
messages = []

add_user_message(
    messages,
    # "Create and save a fake computer science article",
    """
    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    The buggy system generated this malformed output when calling save_article:
    [Generate the exact malformed output here that includes "word_count": undefined]
    This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """,
)

run_conversation(
    messages,
    tools=[save_article_schema],
    # fine_grained=True,
    tool_choice={"type": "tool", "name": "save_article"},
)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Cb6AmZqyPL9zPPXJue4Ko'}